# 리포트 수정 후 재검토 — 수정 반영 상태와 남은 평가 계약

> ### 한 일
> **수정된 소스와 보고서를 대조하고 수치 반례를 CPU에서 다시 계산했다.**

### 결과
1. 경량 검사 6 [^1]항목이 각 명시 조건을 통과했다.
2. 문턱 공유·순위·OFDM·지면 조건과 주요 비교군 설계의 문구가 개선됐다.
3. 속도 표준편차를 9 [^2]배 작게 쓰던 오류는 원 생성기에서 고쳐졌다(R39) — 원장이 이제 물리 단위다.
4. 논문 조각과 이전 검토 보고서에는 수정 전 판단이 남아 있다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 계산 | 직접 국소 합 CFAR 대조, 변하는 프레임의 표적 피크, 물리 단위 Jacobian 역행렬 |
| 범위 | 선별 파일의 수정 반영과 설계 검토. RF 측정·GPU 실행·전체 원장 재생성은 별도 작업이다. |

### 재현

```bash
/workspace/.venvs/py312/bin/python benchmark/review_fixes_followup_0916.py
```

| | |
|---|---|
| 출력 | `outputs/fixes_followup_0916.json` |
| 소요 | CPU 단일 코어 경량 계산 |

---

## 개선된 부분

| 항목 | 현재 판단 |
|---|---|
| 공유 SNR 문턱 | 먼저 얻은 파형의 값을 공유한다는 가정과 NR 자체 문턱 공백을 명시했다. |
| 기체 순위·상관 | 단일 자세와 자세평균의 순위를 분리하고 더 큰 상관 열을 원장에서 읽는다. |
| 관측가능성 본편 | 국소 상태 랭크·관측가능 부분공간 CRLB·전역 모호성을 구분했다. |
| 계획 | OFDM 잡음, 지면 조건, 원자료 평가, 동일 후보 로터 비교, 감시 채널 배분, 송신 자원 개입, 동일 하드웨어 배치 비교를 구체화했다. |

문구와 설계의 수정 상태이며 실외 성능을 얻었다는 뜻과 구분한다.

근거: [src/build_part10_results.py](../src/build_part10_results.py) 행 753 [^3] · [src/build_part10_results.py](../src/build_part10_results.py) 행 1282 [^4] · [src/build_part05_anchor.py](../src/build_part05_anchor.py) 행 92 [^5] · [src/build_part09_detector.py](../src/build_part09_detector.py) 행 707 [^6]

## 구현 수정도 확인한 부분

CPU·RD 크기 [64 [^7], 32 [^8]]·강한 셀 진폭 100000 [^9]·명목 Pfa 0.0001 [^10]에서 직접 국소 합과 현재 CFAR의 불일치는 0 [^11]셀이다.

서로 다른 QPSK 기준 프레임 32 [^12]개·프레임 길이 256 [^13]에서 주입 피크 [19 [^14], 7 [^15]]와 복원 피크 [19 [^16], 7 [^17]]가 일치했다.

프레임별 기준 입력을 선택하는 경로를 검사했다. 반복 프레임용 기본 경로를 사용하는 호출부는 실제 파형 조건과 맞춰야 한다. 과거 결과의 재계산 여부는 이 검사와 별개다.

근거: [src/detection_gpu.py](../src/detection_gpu.py) 행 69 [^18] · [src/passive_process.py](../src/passive_process.py) 행 133 [^19]

## 속도 스케일 정정이 원 계산에 합쳐졌다 (R39)

원장 조건 nr100_G3 [^20]·관측 3 [^21] s·epoch 16 [^22]개·두 수신 위치는 원장에 보존했다.

물리 단위 Jacobian에서 얻은 속도 표준편차는 [0.009365 [^23], 0.01673 [^24], 0.0904 [^25]] m/s, 현재 계산은 [0.009365 [^26], 0.01673 [^27], 0.0904 [^28]] m/s다.

현재 H의 속도 열에 T를 곱하면 좌표는 v/T가 된다. 그 공분산을 속도로 되돌릴 때 T를 곱해야 하지만 코드는 나눈다. 문서대로 vT 좌표를 쓰려면 처음부터 H의 속도 열을 T로 나눠야 한다.

대조 관측시간 1 [^29] s에서는 배율 오류가 숨는다. 해당 full-rank 위치 CRLB(rms)는 0.1896 [^30] m로 유지된다.

이는 새로 발견된 보정식이 아니라 별도 fixups에 이미 기록된 정정의 통합 누락이다. 특이 행렬의 pinv·상대 고유값 판정은 좌표를 일관되게 고친 뒤 따로 재평가한다.

근거: [benchmark/verify_observability.py](../benchmark/verify_observability.py) 행 367 [^31] · [benchmark/verify_observability.py](../benchmark/verify_observability.py) 행 627 [^32] · [benchmark/report4_fixups.py](../benchmark/report4_fixups.py) 행 551 [^33]

## 문구 전파와 검토 보고서의 현재성

논문 조각에는 순간 위치 랭크와 시간 누적 상태 랭크를 연결하고 CRLB를 위치 RMS로 부르는 옛 문장이 남아 있다. 생성기는 축약된 archive notebook에서 논문 조각 생성을 건너뛰며 기존 파일을 보존한다.

앞서 작성한 검토 보고서도 현재 상태에 맞춘 상태 구분이 필요하다. 순위 재계산 조건은 이전 문장의 반례를 확인하지만 현재 문장에 그 주장이 남았는지 검사하지 않는다. 수정된 채널 배분·비교군을 다시 미해결로 출력하는 부분도 있다.

이전 검토는 당시 스냅샷으로 보존하고 항목별 수정·부분 수정·미해결 상태를 별도로 연결한다. 재계산 통과와 수정 반영 완료를 나눠 표시한다.

근거: [docs/paper/04_detector.md](../docs/paper/04_detector.md) 행 43 [^34] · [src/build_part09_detector.py](../src/build_part09_detector.py) 행 864 [^35] · [benchmark/review_research_logic_0916.py](../benchmark/review_research_logic_0916.py) 행 88 [^36] · [benchmark/review_research_logic_0916.py](../benchmark/review_research_logic_0916.py) 행 206 [^37]

## 추가 설계 과제: 추적 연속성의 분모와 성공 조건

중심 질문은 배치에 따른 track continuity 개선으로 좁혀졌다. 다음 단계는 공통 평가 공간·전체 비행시간 분모·truth 매칭 허용오차·ID 변경·최대 coasting·구간 제외 규칙을 고정하는 일이다.

각 배치가 잘 보이는 구간만 분모로 삼으면 좁은 coverage가 유리해질 수 있다. 검출이 끊겨도 예측만 지속하는 tracker에는 위치 매칭이 틀어진 뒤 연속성 점수를 주지 않도록 한다.

주요 지표·최소 의미 효과·false-track 및 위치오차 제약·비행/날짜 단위 표본 수와 불확실성 계획을 함께 적는다. 현행 계획은 효과 크기 미정을 솔직하게 표시한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 205 [^38]

## 추가 설계 과제: 트윈이 배치를 고르는 규칙

같은 후보와 하드웨어를 쓰는 비교군은 마련됐다. 이제 상대 RT 출력에서 후보 점수로 가는 식, 예상 궤적의 가중치, 통신 제약과 설정 허용오차를 고정한다.

자유공간 일치 → rotor 대비 → 배치 순위 → 실제 추적 연속성은 서로 다른 평가 단계다. 평가 비행 전에 순위를 예측하고 선택 배치의 성능과 실측 최선 후보 대비 손실을 함께 보고한다.

실측 최선 후보를 평가 뒤 구하면 oracle 진단으로 표시하고 그 추가 탐색 비용을 실용 measured-search baseline의 보정 예산과 구분한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 168 [^39] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 205 [^38]

## 추가 설계 과제: holdout과 배치 비교의 교란 통제

다른 날짜·궤적을 평가에 쓰면 설정 선택의 누설을 줄인다. 장비를 옮겨 순서대로 측정하는 배치 비교에서는 날짜·바람·간섭·비행 조건 차이를 추가로 통제해야 한다.

공통 날짜/세션 블록 안에서 각 방법에 같은 궤적군을 반복하고 순서를 무작위화하거나 균형 배치한다. 고정 기준 배치를 다시 측정해 drift를 기록하고 블록 내 성능 차이를 비교한다.

사이트 일반화는 새 사이트별 보정 허용과 무보정 전이를 구분해 정의한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 213 [^40]

## 기존 시뮬레이션의 해석 범위

truth에 붙인 거리축과 steering vector 길이에 따른 이상적 배열 이득은 현재 소스가 제한으로 명시한다. 이 단순화를 실제 거리·방위·추적 정확도로 인용하는 경우 별도의 독립 수신 처리와 truth 사후 평가가 필요하다.

근거: [src/experiment_detection.py](../src/experiment_detection.py) 행 25 [^41]

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 속도 좌표계 정정을 원 생성기에 통합하고 관련 원장을 다시 계산한다 | 원장과 보정 파일 사이의 단위 일관성 | 관측가능성 계산 |
| 논문 조각과 검토 보고서에 최신 상태를 연결한다 | 재빌드 뒤에도 독자가 같은 범위의 주장을 읽는다 | 논문 생성 경로와 검토 이력 |
| 연속성 정의·배치 점수·블록 실험 순서를 고정한다 | 측정 전에 주효과와 인과 비교가 결정된다 | 현행 pipeline 평가 계약 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 41개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/fixes_followup_0916.json` | `check_count` | 6 |
| [^2] | `outputs/fixes_followup_0916.json` | `velocity.factor` | 9 |
| [^3] | `outputs/fixes_followup_0916.json` | `sources.threshold.line` | 753 |
| [^4] | `outputs/fixes_followup_0916.json` | `sources.rank.line` | 1282 |
| [^5] | `outputs/fixes_followup_0916.json` | `sources.correlation.line` | 92 |
| [^6] | `outputs/fixes_followup_0916.json` | `sources.observability.line` | 707 |
| [^7] | `outputs/fixes_followup_0916.json` | `cfar.shape[0]` | 64 |
| [^8] | `outputs/fixes_followup_0916.json` | `cfar.shape[1]` | 32 |
| [^9] | `outputs/fixes_followup_0916.json` | `cfar.strong_amplitude` | 100000 |
| [^10] | `outputs/fixes_followup_0916.json` | `cfar.pfa` | 0.0001 |
| [^11] | `outputs/fixes_followup_0916.json` | `cfar.mismatched_cells` | 0 |
| [^12] | `outputs/fixes_followup_0916.json` | `reference.frames` | 32 |
| [^13] | `outputs/fixes_followup_0916.json` | `reference.frame_length` | 256 |
| [^14] | `outputs/fixes_followup_0916.json` | `reference.expected_peak[0]` | 19 |
| [^15] | `outputs/fixes_followup_0916.json` | `reference.expected_peak[1]` | 7 |
| [^16] | `outputs/fixes_followup_0916.json` | `reference.actual_peak[0]` | 19 |
| [^17] | `outputs/fixes_followup_0916.json` | `reference.actual_peak[1]` | 7 |
| [^18] | `outputs/fixes_followup_0916.json` | `sources.cfar.line` | 69 |
| [^19] | `outputs/fixes_followup_0916.json` | `sources.reference.line` | 133 |
| [^20] | `outputs/fixes_followup_0916.json` | `velocity.cell` | nr100_G3 |
| [^21] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].duration_s` | 3 |
| [^22] | `outputs/fixes_followup_0916.json` | `velocity.epochs` | 16 |
| [^23] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].physical_velocity_std_ms[0]` | 0.009365 |
| [^24] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].physical_velocity_std_ms[1]` | 0.01673 |
| [^25] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].physical_velocity_std_ms[2]` | 0.0904 |
| [^26] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].current_velocity_std_ms[0]` | 0.009365 |
| [^27] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].current_velocity_std_ms[1]` | 0.01673 |
| [^28] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].current_velocity_std_ms[2]` | 0.0904 |
| [^29] | `outputs/fixes_followup_0916.json` | `velocity.rows[0].duration_s` | 1 |
| [^30] | `outputs/fixes_followup_0916.json` | `velocity.rows[1].physical_position_rms_m` | 0.1896 |
| [^31] | `outputs/fixes_followup_0916.json` | `sources.scale.line` | 367 |
| [^32] | `outputs/fixes_followup_0916.json` | `sources.unscale.line` | 627 |
| [^33] | `outputs/fixes_followup_0916.json` | `sources.known_fix.line` | 551 |
| [^34] | `outputs/fixes_followup_0916.json` | `sources.paper.line` | 43 |
| [^35] | `outputs/fixes_followup_0916.json` | `sources.paper_builder.line` | 864 |
| [^36] | `outputs/fixes_followup_0916.json` | `sources.review.line` | 88 |
| [^37] | `outputs/fixes_followup_0916.json` | `sources.stale_review.line` | 206 |
| [^38] | `outputs/fixes_followup_0916.json` | `sources.endpoint.line` | 205 |
| [^39] | `outputs/fixes_followup_0916.json` | `sources.placement.line` | 168 |
| [^40] | `outputs/fixes_followup_0916.json` | `sources.holdout.line` | 213 |
| [^41] | `outputs/fixes_followup_0916.json` | `sources.ideal_scope.line` | 25 |